<a href="https://colab.research.google.com/github/shashank3110/genAI/blob/main/business_assistant_langraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4


In [1]:
pip install -U langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00


In [3]:
import os
os.environ["GOOGLE_API_KEY"]="" # ADD API key

In [5]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph
from typing import TypedDict, List
from langchain_core.messages import BaseMessage
import pandas as pd

## Load & Explore knowledge base

In [6]:
df_b = pd.read_csv('/content/clothing_business_dataset.csv')
df_s = pd.read_csv('/content/clothing_schema_dataset.csv')


In [7]:
df_b.head()

,Category,Name,Description
0,Fact Table,Fact_Sales,Tracks product-level sales transactions
1,Fact Table,Fact_Inventory,Tracks stock levels per product/location
2,Fact Table,Fact_Orders,Tracks order lifecycle
3,Fact Table,Fact_Returns,Tracks returned items
4,Fact Table,Fact_Marketing_Performance,Tracks campaign performance


In [8]:
df_b['Category'].unique()

array(['Fact Table', 'Dimension Table', 'KPI', 'Stakeholder'],
      dtype=object)

In [9]:
df_b[df_b['Category']=='KPI']

,Category,Name,Description
11,KPI,Revenue,Total sales before returns
12,KPI,Gross Profit,Revenue minus cost
13,KPI,Conversion Rate,Orders divided by visitors
14,KPI,Inventory Turnover,COGS divided by avg inventory
15,KPI,Return Rate,Returned items divided by sold items
16,KPI,Customer Lifetime Value,Total revenue per customer


In [10]:
df_b[df_b['Category']=='Stakeholder']

,Category,Name,Description
17,Stakeholder,Sales Department,Sales roles and KPIs
18,Stakeholder,Supply Chain,Inventory and logistics roles
19,Stakeholder,Marketing,Campaign and growth roles
20,Stakeholder,Finance,Financial oversight roles
21,Stakeholder,Product/Merchandising,Product and design roles


In [27]:
df_s.head(10)

,table_name,column_name,data_type,key_type
0,Fact_Sales,sales_id,BIGINT,PK
1,Fact_Sales,order_id,BIGINT,FK
2,Fact_Sales,product_id,INT,FK
3,Fact_Sales,total_amount,"DECIMAL(10,2)",NaN
4,Dim_Product,product_id,INT,PK
5,Dim_Product,product_name,VARCHAR(255),NaN
6,Dim_Product,category,VARCHAR(100),NaN
7,Dim_Customer,customer_id,INT,PK
8,Dim_Customer,first_name,VARCHAR(100),NaN
9,Dim_Customer,country,VARCHAR(100),NaN


(10, 4)

In [12]:
# df_s[df_s['table_name']=='Stakeholder']
df_s['table_name'].unique()

array(['Fact_Sales', 'Dim_Product', 'Dim_Customer'], dtype=object)

## Initialize LLM

In [13]:

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # or gemini-1.5-flash
    temperature=0
)

In [15]:
llm.invoke(f'You are business assitant and uou answer business questions using given context. [Question]: Who are the key stakeholders for the business? . [Context] {df_b} ')

AIMessage(content='Based on the provided context, the key stakeholders for the business are:\n\n*   Sales Department\n*   Supply Chain\n*   Marketing\n*   Finance\n*   Product/Merchandising', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d7c5b-ab13-78a0-94fc-19ad482a5f2b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 445, 'output_tokens': 83, 'total_tokens': 528, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 43}})

In [28]:
# Define tools

# def rag_answer(question: str, context: pd.DataFrame) -> int:
#     """
#     Answer User's business questions using provided Context.

#     Args:
#         question: User question: str
#         context: Business context highlighting products,
#                   kpis, stakeholders, tables, etc : pandas dataframe

#     """

@tool
def rag_answer(question: str) -> str:
    """
    Answer User's business questions using provided Context.

    Args:
        question: User question: str

    """

    prompt = f"""You are business FAQ assitant and you answer business questions
                  using given context. [Question]: {question} . [Context] {df_b} """


    return llm.invoke(prompt)


@tool
def sql_answer(question: str) -> str:
    """
    Generate SQL code to answer User's business questions using provided Context.

    Args:
        question: User question: str

    """

    prompt = f"""You are SQL assitant and you answer business questions with valid SQL code
                  using given context which contains the schema for the various
                  tables. [Question]: {question} . [Context] {df_s} """


    return llm.invoke(prompt)


# def divide(a: int, b: int) -> float:
#     """Divide `a` and `b`.

#     Args:
#         a: First int
#         b: Second int
#     """
#     return a / b

# def power(a: int, b: int) -> float:
#     """Calculates exponents: a to power b

#     Args:
#         a: First int
#         b: Second int
#     """

#     return a**b



In [29]:
# initialize tools
tools = [rag_answer, sql_answer]
tool_names = [tool.name for tool in tools]
llm_with_tools = llm.bind_tools(tools)

In [19]:
class AgentState(MessagesState):
    pass



In [30]:

def agent(state: AgentState):
    if not state["messages"]:
        raise ValueError("Empty state!")

    response = llm_with_tools.invoke(state["messages"])
    print(f"[DEBUG] MODEL OUTPUT: {response}")
    print("[DEBUG] TOOL CALLS:", getattr(response, "tool_calls", None))

    return {"messages":  state["messages"] + [response]}


In [21]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)


In [22]:
# for conditional branching/routing


def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return "end"


In [ ]:
#DEBUG CODE
def tool_node_debug(state):
    print("\n🔥 TOOL NODE INPUT:", state)

    result = tool_node.invoke(state)

    print("\n🔥 TOOL NODE OUTPUT:", result)

    return result

In [ ]:
# DEBIUG CODE
def debug_node(name, fn):
    def wrapper(state):
        print(f"\n🔥 {name} INPUT:")
        for m in state["messages"]:
            print(type(m), m)

        result = fn(state)

        print(f"\n🔥 {name} OUTPUT:")
        for m in result.get("messages", []):
            print(type(m), m)

        return result
    return wrapper


## Build Agent execution control graph

In [31]:
business_graph = StateGraph(AgentState)
business_graph.add_node("agent", agent)
business_graph.add_node("tools",tool_node)
# math_graph.add_node("tools",tool_node_debug)

# math_graph.add_node("agent", debug_node("AGENT", agent))
# math_graph.add_node("tools", debug_node("TOOLS", tool_node_debug))


business_graph.set_entry_point("agent")

business_graph.add_conditional_edges("agent", should_continue,  {
        "tools": "tools",
        "end": "__end__"
    }
    )

business_graph.add_edge("tools", "agent")
# math_graph.add_edge("tools", "end")

In [32]:
# compile agentic graph
business_agent = business_graph.compile()

In [25]:
from langchain_core.messages import HumanMessage, SystemMessage


In [37]:
result = business_agent.invoke(

     {
          "messages": [
              SystemMessage(content="You are a Business assistant who answers business related questions through text or SQL code. Always use tools for answering business FAQ questions or providing SQL code."),
              HumanMessage(content="who are the key stakeholders?"),
              HumanMessage(content="How can i calculate gross profit?"),
              HumanMessage(content="Define inventory turnover and customer life time value?"),
              HumanMessage(content="Calculate total sales per product "),
              # HumanMessage(content=" Get total customers per country or region?")

              ]
     }

)

[DEBUG] MODEL OUTPUT: content='' additional_kwargs={'function_call': {'name': 'sql_answer', 'arguments': '{"question": "Calculate total sales per product"}'}, '__gemini_function_call_thought_signatures__': {'324f8e94-2bc1-4076-902d-8833dddc518e': 'CtYDAb4+9vsF8b5lKRvqrT+BXyQ+U0HDE6e5Pfvu54YYp/zeuHZc4UZ19sA/oQYFLDqpOLANKc6Hl2veCusZrQrdiiUb0yahTBkEd/To4+n4rDoLZ+7haC/o2JBoJsoY++YzwBNRMQkDBCqHHsgSeA+0wIipPshBJHU+UbuS/3Mbs3u5EEW2plUzLkv55nHrvh9mSDki0qnmNEGLkxSDqJSeVOzHQK03ERlZuawZ3YSCLRHPoT6j8JMJMkWq6QY7CG0zLgeoXrB33iDxmYARbCEgXQCwZenJGjLwVUPZb1JTArKGCRxT/L64EwPryh92BfzMKlSlsXLVwwmNuWpmAqrRNElB6ZZsg+H7Vxm5JlafZRBmtOEIS1HQZVe/Zq3eKrY4GAau7BAZ3+pnDQYP2RNwOQbUfiT1iHkjLpF2sm2bg73SS3OHab6kdRQjm6Bagpb5Ts8KeCyAOHNjlXpCR0KjNy1wnrR+KGYiWJ0SkSG+UIHXwd0VNq1CPRZFCsd5w+QK999ppoIK/FHhYqSwRukGs4b/7j2+H6K0x8l9fF7d67OtRuLnLOo0sVwoALEQVRLXuh5iG6pqptZrSOkadAByX38oVQUYHOAk98RKb1xxnTsqDBbgCBE='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider'

In [38]:
print(result["messages"][-1].content)

['The key stakeholders are:\n* Sales Department\n* Supply Chain\n* Marketing\n* Finance\n* Product/Merchandising\n\nGross Profit is calculated as **Revenue minus cost**.\n\n*   **Inventory Turnover** is defined as **COGS divided by avg inventory**.\n*   **Customer Lifetime Value** is defined as **Total revenue per customer**.', '```sql\nSELECT\n  dp.product_name,\n  SUM(fs.total_amount) AS total_sales\nFROM Fact_Sales AS fs\nJOIN Dim_Product AS dp\n  ON fs.product_id = dp.product_id\nGROUP BY\n  dp.product_name\nORDER BY\n  total_sales DESC;\n```']


In [43]:

for answers in result["messages"][-1].content:

  for answer in answers.split('\n\n'):
    print('-------')
    print(answer)
    print('-------')

-------
The key stakeholders are:
* Sales Department
* Supply Chain
* Marketing
* Finance
* Product/Merchandising
-------
-------
Gross Profit is calculated as **Revenue minus cost**.
-------
-------
*   **Inventory Turnover** is defined as **COGS divided by avg inventory**.
*   **Customer Lifetime Value** is defined as **Total revenue per customer**.
-------
-------
```sql
SELECT
  dp.product_name,
  SUM(fs.total_amount) AS total_sales
FROM Fact_Sales AS fs
JOIN Dim_Product AS dp
  ON fs.product_id = dp.product_id
GROUP BY
  dp.product_name
ORDER BY
  total_sales DESC;
```
-------


In [ ]:
# result